# RAG Colab Notebook
This single notebook runs in Google Colab. It installs dependencies and provides an interactive `ipywidgets` UI to upload a PDF, choose a domain (Media, Law, Telecom, General), and either ask questions or generate a structured summary using a Retrieval-Augmented Generation (RAG) pipeline.

Run the first code cell to install dependencies and set your `OPENAI_API_KEY` value (in Cell 1). Then run the second cell to show the UI.

In [ ]:
# Colab Cell 1: Install dependencies
# Run this cell first. Do NOT pin langchain to 0.3.x — Colab pre-installs
# langchain-classic / langgraph which require langchain-core >= 1.4.4.

!pip install -q \
  langchain \
  langchain-openai \
  langchain-community \
  langchain-chroma \
  langchain-text-splitters \
  chromadb \
  pypdf \
  ipywidgets \
  sentence-transformers

# Enable ipywidgets rendering in Colab
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

# Print installed versions for verification
try:
    import importlib.metadata as _meta
except ImportError:
    import importlib_metadata as _meta

for _pkg in [
    "langchain", "langchain-core", "langchain-openai",
    "langchain-chroma", "langchain-text-splitters",
    "chromadb", "pypdf", "ipywidgets", "sentence-transformers",
]:
    try:
        print(f"{_pkg}=={_meta.version(_pkg)}")
    except Exception:
        print(f"{_pkg} not found")

print("\nInstallation complete. Run Cell 2 to launch the UI.")
print("You will enter your OpenAI API key directly in the UI.")

In [ ]:
# Colab Cell 2: RAG app with ipywidgets UI
# Run after Cell 1. Enter your OpenAI API key in the UI field below.

import os
import io
import html as _html
import threading
from pypdf import PdfReader
from IPython.display import display
import ipywidgets as widgets
from uuid import uuid4

# Re-enable widget manager in case Cell 1 wasn't re-run
try:
    from google.colab import output as _colab_out
    _colab_out.enable_custom_widget_manager()
except Exception:
    pass

# Modern LangChain imports (compatible with langchain-core >= 1.4.4)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Utility ───────────────────────────────────────────────────────────────────

def pdf_bytes_to_text(pdf_bytes):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    pages = []
    for p in reader.pages:
        try:
            pages.append(p.extract_text() or "")
        except Exception:
            pages.append("")
    return "\n\n".join(pages)

def get_api_key():
    return api_key_input.value.strip()

def get_llm():
    return ChatOpenAI(model="gpt-4o-mini", openai_api_key=get_api_key(), temperature=0.0)

def build_vectorstore_with_files(files, existing_vectordb=None, persist_dir=None):
    """Index list of (fname, bytes). Adds to existing_vectordb when provided."""
    embedding_fn = OpenAIEmbeddings(
        model="text-embedding-3-small", openai_api_key=get_api_key()
    )
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    all_texts, all_metadatas = [], []
    for fname, pdf_bytes in files:
        chunks = splitter.split_text(pdf_bytes_to_text(pdf_bytes))
        all_texts.extend(chunks)
        all_metadatas.extend([{"source": fname}] * len(chunks))
    if existing_vectordb is None:
        persist_dir = persist_dir or f"chroma_store_{uuid4().hex}"
        vectordb = Chroma.from_texts(
            texts=all_texts,
            embedding=embedding_fn,
            metadatas=all_metadatas,
            persist_directory=persist_dir,
        )
    else:
        existing_vectordb.add_texts(texts=all_texts, metadatas=all_metadatas)
        vectordb = existing_vectordb
    return vectordb.as_retriever(search_kwargs={"k": 5}), vectordb, persist_dir

# ── Prompts ───────────────────────────────────────────────────────────────────

_NOT_FOUND = "I cannot find that information in the provided document."

BASE_PROMPT = f"""You are a helpful assistant. Answer ONLY from the CONTEXT below.
Domain: {{domain}}

Rules:
- Use only the CONTEXT to answer. If not found, respond exactly: "{_NOT_FOUND}"
- Be concise. For "Law" emphasize clauses/liabilities; "Telecom"/"Media" emphasize specs/metrics; "General" give neutral answers; for custom domains apply relevant expertise.

CONTEXT:
{{context}}

QUESTION: {{question}}

Answer:"""

SUMMARY_PROMPT = """You are a summarization assistant. Summarize ONLY from the CONTEXT below.
Domain: {domain}
If details are missing, note it briefly.

CONTEXT:
{context}

Produce a structured summary with headings."""

QA_TEMPLATE = PromptTemplate(template=BASE_PROMPT, input_variables=["context", "question", "domain"])
SUMMARY_TEMPLATE = PromptTemplate(template=SUMMARY_PROMPT, input_variables=["context", "domain"])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# ── RAG functions ─────────────────────────────────────────────────────────────

def query_with_rag(retriever, question, domain):
    llm = get_llm()
    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
            "domain": lambda _: domain,
        }
        | QA_TEMPLATE
        | llm
        | StrOutputParser()
    )
    result = chain.invoke(question)

    if _NOT_FOUND not in result:
        return result

    # Answer not in document → fall back to LLM general knowledge
    fallback = llm.invoke(
        f"Question: {question}\n\n"
        "The user's uploaded document did not contain this information. "
        "Answer from your general knowledge. Be helpful and accurate."
    )
    fallback_text = fallback.content if hasattr(fallback, "content") else str(fallback)
    return (
        "⚠️ **Not found in uploaded document(s)** — answering from general knowledge:\n\n"
        + fallback_text
    )

def generate_summary(retriever, domain):
    llm = get_llm()
    docs = retriever.invoke("summary overview key points")
    combined = "\n\n".join(d.page_content for d in docs)
    result = llm.invoke(SUMMARY_TEMPLATE.format(context=combined, domain=domain))
    return result.content if hasattr(result, "content") else str(result)

# ── State ─────────────────────────────────────────────────────────────────────

_state = {
    "vectordb": None,
    "retriever": None,
    "persist_dir": None,
    "indexed_files": [],
}

# ── Answer display helper (HTML widget — thread-safe, no display() needed) ────

def _md_to_html(text):
    """Convert a limited subset of Markdown to HTML for the answer widget."""
    import re
    lines = text.split("\n")
    out = []
    for line in lines:
        # headings
        if line.startswith("### "):
            out.append(f"<h4 style='margin:8px 0 4px'>{_html.escape(line[4:])}</h4>")
        elif line.startswith("## "):
            out.append(f"<h3 style='margin:10px 0 4px'>{_html.escape(line[3:])}</h3>")
        elif line.startswith("# "):
            out.append(f"<h2 style='margin:12px 0 4px'>{_html.escape(line[2:])}</h2>")
        elif line.startswith("- ") or line.startswith("* "):
            out.append(f"<li>{_html.escape(line[2:])}</li>")
        elif line.strip() == "":
            out.append("<br>")
        else:
            safe = _html.escape(line)
            # bold & italic inline
            safe = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", safe)
            safe = re.sub(r"\*(.+?)\*", r"<i>\1</i>", safe)
            # preserve ⚠️ literally
            out.append(f"<p style='margin:4px 0'>{safe}</p>")
    return "".join(out)

# ── Widgets ───────────────────────────────────────────────────────────────────

# Step 1 — API key
api_key_input = widgets.Password(
    value="", placeholder="sk-...", description="OpenAI Key:",
    layout=widgets.Layout(width="400px"),
)
api_key_status = widgets.HTML('<span style="color:orange">Enter your OpenAI API key</span>')

def _on_key_change(change):
    key = change["new"].strip()
    api_key_status.value = (
        '<span style="color:green">&#10003; Key set</span>'
        if key.startswith("sk-") and len(key) > 20
        else '<span style="color:orange">Enter a valid sk-... key</span>'
    )
api_key_input.observe(_on_key_change, names="value")

# Step 2 — Category selector
_DEFAULT_DOMAINS = ["Media", "Law", "Telecom", "General"]
domain_dropdown = widgets.Dropdown(
    options=_DEFAULT_DOMAINS, value="General", description="Category:",
    layout=widgets.Layout(width="220px"),
)
new_category_input = widgets.Text(
    value="", placeholder="e.g. Finance, Healthcare…", description="",
    layout=widgets.Layout(width="210px"),
)
add_category_btn   = widgets.Button(description="+ Add",    button_style="warning", layout=widgets.Layout(width="70px"))
remove_category_btn = widgets.Button(description="✕ Remove", button_style="danger",  layout=widgets.Layout(width="90px"))
category_msg = widgets.HTML("")

def on_add_category(b):
    name = new_category_input.value.strip()
    if not name:
        category_msg.value = '<span style="color:orange">Type a name first.</span>'; return
    opts = list(domain_dropdown.options)
    if name in opts:
        category_msg.value = f'<span style="color:orange">"{name}" already exists.</span>'; return
    opts.append(name); domain_dropdown.options = opts; domain_dropdown.value = name
    new_category_input.value = ""
    category_msg.value = f'<span style="color:green">&#10003; Added "{name}"</span>'
add_category_btn.on_click(on_add_category)

def on_remove_category(b):
    current = domain_dropdown.value
    if current in _DEFAULT_DOMAINS:
        category_msg.value = f'<span style="color:orange">Cannot remove built-in "{current}".</span>'; return
    opts = [o for o in domain_dropdown.options if o != current]
    domain_dropdown.options = opts; domain_dropdown.value = opts[-1] if opts else None
    category_msg.value = f'<span style="color:green">&#10003; Removed "{current}"</span>'
remove_category_btn.on_click(on_remove_category)

# Step 2 — Upload area (google.colab.files.upload inside Output widget)
upload_out = widgets.Output(layout=widgets.Layout(
    border="1px dashed #555", padding="6px", min_height="30px", margin="4px 0",
))
upload_btn = widgets.Button(
    description="📂 Upload PDF(s)", button_style="info",
    layout=widgets.Layout(width="165px"),
    tooltip="Click to open a file picker. You can click again later to add more docs.",
)
clear_btn = widgets.Button(
    description="Clear Index", button_style="danger",
    layout=widgets.Layout(width="110px"),
    tooltip="Remove all indexed documents",
)
indexed_files_display = widgets.HTML('<i style="color:#888">No documents indexed yet</i>')

def _refresh_indexed_display():
    files = _state["indexed_files"]
    if not files:
        indexed_files_display.value = '<i style="color:#888">No documents indexed yet</i>'
        return
    rows = "".join(f'<li style="color:green;margin:2px 0">&#10003; {_html.escape(f)}</li>' for f in files)
    count = len(files)
    indexed_files_display.value = (
        f'<b>Indexed ({count} doc{"s" if count != 1 else ""}):</b>'
        f'<ul style="margin:4px 0;padding-left:18px">{rows}</ul>'
        f'<span style="font-size:12px;color:#aaa">Click "📂 Upload PDF(s)" again to add more documents.</span>'
    )

# Status and answer — HTML widgets (thread-safe, no display() required)
status_html = widgets.HTML('<i style="color:#888">Ready.</i>')
answer_html  = widgets.HTML(
    '<div style="color:#888;font-style:italic;padding:8px">No answer yet.</div>'
)

def _set_status(msg, color="#ccc"):
    status_html.value = f'<span style="color:{color}">{_html.escape(msg)}</span>'

def _set_answer(text):
    """Render answer markdown as HTML — called from background thread safely."""
    rendered = _md_to_html(text)
    answer_html.value = (
        f'<div style="font-family:sans-serif;font-size:14px;'
        f'line-height:1.7;padding:10px;border:1px solid #555;border-radius:4px">'
        f'{rendered}</div>'
    )

def _index_files_async(pdf_files):
    names = [fn for fn, _ in pdf_files]
    _set_status(f"Indexing {len(pdf_files)} file(s): {', '.join(names)}…", "#aaa")
    def worker():
        try:
            retriever, vectordb, persist_dir = build_vectorstore_with_files(
                pdf_files,
                existing_vectordb=_state["vectordb"],
                persist_dir=_state["persist_dir"],
            )
            _state.update({"vectordb": vectordb, "retriever": retriever, "persist_dir": persist_dir})
            for fn in names:
                if fn not in _state["indexed_files"]:
                    _state["indexed_files"].append(fn)
            _refresh_indexed_display()
            count = len(_state["indexed_files"])
            _set_status(f"✓ {count} doc(s) indexed and ready. Ask a question or generate a summary.", "lightgreen")
        except Exception as e:
            _set_status(f"Indexing failed: {e}", "tomato")
    threading.Thread(target=worker, daemon=True).start()

def on_upload_btn_click(b):
    if not get_api_key():
        _set_status("Please enter your OpenAI API key first.", "orange"); return
    upload_out.clear_output()
    try:
        from google.colab import files as _gfiles
    except ImportError:
        with upload_out:
            print("Not running in Colab — cannot open file picker.")
        return
    with upload_out:
        print("Opening file picker — select one or more PDFs…")
        try:
            raw = _gfiles.upload()
        except Exception as e:
            print("Upload error:", e); return
    if not raw:
        upload_out.clear_output()
        with upload_out:
            print("No files uploaded.")
        return
    pdf_files = [(fn, bytes(c)) for fn, c in raw.items() if fn.lower().endswith(".pdf")]
    skipped    = [fn for fn in raw if not fn.lower().endswith(".pdf")]
    upload_out.clear_output()
    if skipped:
        _set_status(f"Skipped non-PDF: {', '.join(skipped)}", "orange")
    if not pdf_files:
        _set_status("No valid PDF files found — please upload .pdf files.", "orange"); return
    _index_files_async(pdf_files)
upload_btn.on_click(on_upload_btn_click)

def on_clear_btn_click(b):
    _state.update({"vectordb": None, "retriever": None, "persist_dir": None, "indexed_files": []})
    _refresh_indexed_display()
    answer_html.value = '<div style="color:#888;font-style:italic;padding:8px">No answer yet.</div>'
    _set_status("Index cleared. Upload new PDF(s) to start again.", "#aaa")
clear_btn.on_click(on_clear_btn_click)

# Step 3 — Ask
ask_text   = widgets.Text(value="", description="Question:", layout=widgets.Layout(width="70%"))
ask_button = widgets.Button(description="Ask", button_style="primary")

def on_ask_clicked(b):
    if not get_api_key():
        _set_status("Please enter your OpenAI API key first.", "orange"); return
    if _state["retriever"] is None:
        _set_status("No documents indexed. Upload a PDF first.", "orange"); return
    question = ask_text.value.strip()
    if not question:
        _set_status("Please type a question.", "orange"); return
    domain = domain_dropdown.value
    _set_status(f"Searching {len(_state['indexed_files'])} doc(s) for an answer…", "#aaa")
    answer_html.value = '<div style="color:#aaa;padding:8px"><i>Thinking…</i></div>'
    def worker():
        try:
            ans = query_with_rag(_state["retriever"], question, domain)
            _set_answer(ans)
            _set_status("Done.", "lightgreen")
        except Exception as e:
            _set_status(f"Query failed: {e}", "tomato")
            answer_html.value = f'<div style="color:tomato;padding:8px">Error: {_html.escape(str(e))}</div>'
    threading.Thread(target=worker, daemon=True).start()
ask_button.on_click(on_ask_clicked)

# Step 3 — Summary
summary_button = widgets.Button(description="Generate Summary", button_style="info")

def on_summary_clicked(b):
    if not get_api_key():
        _set_status("Please enter your OpenAI API key first.", "orange"); return
    if _state["retriever"] is None:
        _set_status("No documents indexed. Upload a PDF first.", "orange"); return
    domain = domain_dropdown.value
    count  = len(_state["indexed_files"])
    _set_status(f"Generating summary across {count} doc(s) (domain: {domain})…", "#aaa")
    answer_html.value = '<div style="color:#aaa;padding:8px"><i>Generating summary…</i></div>'
    def worker():
        try:
            summ = generate_summary(_state["retriever"], domain)
            _set_answer(summ)
            _set_status("Summary ready.", "lightgreen")
        except Exception as e:
            _set_status(f"Summary failed: {e}", "tomato")
            answer_html.value = f'<div style="color:tomato;padding:8px">Error: {_html.escape(str(e))}</div>'
    threading.Thread(target=worker, daemon=True).start()
summary_button.on_click(on_summary_clicked)

# ── Layout ────────────────────────────────────────────────────────────────────

key_box = widgets.VBox([
    widgets.HTML("<b>Step 1 — Enter your OpenAI API Key</b>"),
    widgets.HBox([api_key_input, api_key_status]),
])

upload_box = widgets.VBox([
    widgets.HTML("<b>Step 2 — Choose Category &amp; Upload PDF(s)</b>"),
    domain_dropdown,
    widgets.HBox([new_category_input, add_category_btn, remove_category_btn]),
    category_msg,
    widgets.HTML("<div style='margin-top:6px'></div>"),
    widgets.HBox([upload_btn, clear_btn]),
    upload_out,
    indexed_files_display,
])

action_box = widgets.VBox([
    widgets.HTML("<b>Step 3 — Ask or Summarize</b>"),
    widgets.HTML("<span style='font-size:12px;color:#aaa'>You can ask multiple questions and generate summaries anytime.</span>"),
    widgets.HBox([ask_text, ask_button]),
    widgets.HTML("<i>— or —</i>"),
    summary_button,
])

ui = widgets.VBox([
    key_box,
    widgets.HTML("<hr>"),
    widgets.HBox([upload_box, action_box]),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Status</b>"),
    status_html,
    widgets.HTML("<b style='display:block;margin-top:10px'>Answer</b>"),
    answer_html,
])

display(ui)